# 04 · Diễn giải itemset bằng bằng chứng ảnh

**Mục đích:** nối 3–4 itemset Set được Notebook 03 tự động chọn tới centroid, patch nguồn, ảnh hỗ trợ, ảnh đối chứng và phân bố lớp.

**Điều kiện chạy:** Notebook 02 có TC7 manifest; Notebook 03 đã hoàn tất mining, parity audit và selector, tạo `selected_itemsets.csv`; archive ảnh/nhãn đã xác minh.

**Đầu vào:** các manifest/hash trong run root và danh sách đại diện tự động của Notebook 03.

**Đầu ra:** gallery, retrieval CSV, class statistics, phiếu annotations và báo cáo có trạng thái `PENDING_REVIEW` cho tới khi reviewer nhập nhận xét.

## 1. Chọn run và kiểm tra handoff manifests

Cell này tìm duy nhất một run TC7 hoặc dùng `FASHION_RELEASE_RUN_ID`. Kiểm tra TC7, mining manifest, selection manifest và selection CSV tồn tại; xác nhận có đúng 3–4 mẫu trước khi diễn giải.

In [ ]:
using Pkg
using SHA: sha256
using TOML

function find_repo_root(start=pwd())
    current = realpath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src", "EclatFIM.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("Không tìm thấy repository root")
        current = parent
    end
end

function release_runs(repo_root)
    base = joinpath(repo_root, "results", "fashion", "release")
    return [path for path in readdir(base; join=true)
        if isfile(joinpath(path, "transactions", "tc7_manifest.toml"))]
end

repo_root = get(ENV, "FASHION_MNIST_REPO_ROOT", find_repo_root())
Pkg.activate(repo_root)
using EclatFIM
runs = release_runs(repo_root)
requested = get(ENV, "FASHION_RELEASE_RUN_ID", "")
if !isempty(requested)
    RUN_ROOT = joinpath(repo_root, "results", "fashion", "release", requested)
    @assert RUN_ROOT in runs "Không tìm thấy run ID được yêu cầu"
else
    length(runs) == 1 || error("Cần đúng một run TC7 hoặc đặt FASHION_RELEASE_RUN_ID")
    RUN_ROOT = only(runs)
end
TC7_PATH = joinpath(RUN_ROOT, "transactions", "tc7_manifest.toml")
MINING_PATH = joinpath(RUN_ROOT, "mining", "mining_manifest.toml")
SELECTION_MANIFEST_PATH = joinpath(RUN_ROOT, "mining", "selection_manifest.toml")
SELECTED_PATH = joinpath(RUN_ROOT, "mining", "selected_itemsets.csv")
@assert isfile(MINING_PATH) "Chạy Notebook 03 trước"
@assert isfile(SELECTION_MANIFEST_PATH) "Notebook 03 chưa tạo selection manifest"
@assert isfile(SELECTED_PATH) "Notebook 03 chưa tạo danh sách itemset đại diện"
TC7 = TOML.parsefile(TC7_PATH)
MINING = TOML.parsefile(MINING_PATH)
SELECTION = TOML.parsefile(SELECTION_MANIFEST_PATH)
@assert TC7["status"] == "PASS"
@assert TC7["transaction_sha256"] == MINING["transaction_sha256"]
@assert bytes2hex(open(sha256, TC7_PATH)) == MINING["tc7_manifest_sha256"]
@assert bytes2hex(open(sha256, SELECTED_PATH)) == MINING["selected_itemsets_sha256"]
@assert SELECTION["target_selected"] in 3:4
@assert SELECTION["target_selected"] == MINING["selected_itemset_count"]
@assert SELECTION["catalog_sha256"] == MINING["candidate_catalog_sha256"]
@assert length(readlines(SELECTED_PATH)) - 1 == SELECTION["target_selected"]
println("Run: ", basename(RUN_ROOT), " | ", SELECTION["target_selected"],
        " automatic representative itemsets and hashes verified")

## 2. Chạy diễn giải và đối chiếu artifact hashes

Cell này chạy helper diễn giải trên selection, TC7 manifest, mining manifest và archive. Helper xác minh patch assignments, codebook, mapping, metadata, support retrieval và class totals trước khi sinh báo cáo.


In [ ]:
python_candidates = filter(isfile, [
    something(Sys.which("python3"), ""),
    something(Sys.which("python"), ""),
    joinpath(repo_root, ".venv", "bin", "python3"),
    joinpath(repo_root, ".venv", "bin", "python")])
isempty(python_candidates) && error("Không tìm thấy Python 3 trong PATH hoặc .venv/bin")
PYTHON = first(python_candidates)
INTERPRETER = joinpath(EclatFIM.FASHION_MNIST_ROOT, "mining", "scripts", "interpretation.py")
RAW_DIR = joinpath(repo_root, "data", "raw", "fashion-mnist")
println("Python: ", PYTHON)
run(Cmd([PYTHON, INTERPRETER, "--output-dir", RUN_ROOT, "--raw-dir", RAW_DIR]))
SUMMARY_PATH = joinpath(RUN_ROOT, "interpretation", "interpretation_summary.json")
@assert isfile(SUMMARY_PATH)
println(read(SUMMARY_PATH, String))

## 3. Xem gallery ảnh và patch witnesses

Cell này hiển thị gallery HTML do helper tạo. Mỗi card cho thấy visual-word centroid, các patch thật có tọa độ nguồn và tối đa sáu ảnh hỗ trợ cùng một ảnh đối chứng nếu có.


In [ ]:
using Markdown
GALLERY_PATH = joinpath(RUN_ROOT, "interpretation", "pattern_gallery", "index.html")
@assert isfile(GALLERY_PATH)
display(Markdown.HTML(read(GALLERY_PATH, String)))


## 4. Kiểm tra retrieval và phân bố lớp

Cell này đọc summary, retrieval CSV và class statistics. Kiểm tra số ảnh truy xuất bằng support đã khai thác và tổng số hỗ trợ theo lớp bằng cùng support.


In [ ]:
using TOML
retrieval = TOML.parsefile(joinpath(RUN_ROOT, "interpretation", "retrieval_summary.toml"))
@assert retrieval["retrieval_matches_support"] == true
@assert retrieval["retrieved_count"] == retrieval["support_count"]
println("Itemset: ", retrieval["items"])
println("Support / retrieved: ", retrieval["support_count"], " / ", retrieval["retrieved_count"])
println("Class counts: ", retrieval["class_counts"])
println("Retrieval CSV: ", relpath(joinpath(RUN_ROOT, retrieval["csv_path"]), repo_root))


## 5. Xuất phiếu review và báo cáo lượt chạy

Cell này kiểm tra annotations ban đầu và mở báo cáo Markdown. Trước khi reviewer ghi nhận xét, `review_status` phải là `PENDING_REVIEW` và `visual_coherence` là `NA`.


In [ ]:
review_path = joinpath(RUN_ROOT, "interpretation", "pattern_annotations.csv")
report_path = joinpath(RUN_ROOT, "interpretation", "visual_interpretation.md")
manifest = TOML.parsefile(joinpath(RUN_ROOT, "interpretation", "interpretation_manifest.toml"))
@assert manifest["review_status"] == "PENDING_REVIEW"
@assert manifest["visual_coherence"] == "NA"
println(read(report_path, String))
println("Review sheet: ", relpath(review_path, repo_root))


## 6. Tái sinh hình báo cáoCell kế tiếp xuất Hình 6 từ gallery của mẫu `{4,12,24}` và dựng Hình 9 từ toàn bộ 7.228 ứng viên ở minsup 5%. Script xác minh hash của TC7/mining, nối giao dịch với nhãn ảnh, tính lại support từng mẫu và chọn mẫu enrichment cao nhất trong mỗi lớp với tỷ lệ hỗ trợ trong lớp ít nhất 10%. Hình 9 có thêm CSV số liệu để truy vết từng ô màu.

In [11]:
REPORT_FIGURE_SCRIPT = joinpath(repo_root, "src", "applications", "Fashion-Mnist", "mining", "scripts", "report_figures.py")
REPORT_DIR = joinpath(repo_root, "reports", "UD2")
@assert isfile(REPORT_FIGURE_SCRIPT)
run(Cmd([PYTHON, REPORT_FIGURE_SCRIPT, "--run-root", RUN_ROOT, "--report-dir", REPORT_DIR]))

Verified 70,000 transactions, 7,228 candidate itemsets and 10 class labels.
Figure 6: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/reports/UD2/figures/pattern_4_12_24.png
Figure 9: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/reports/UD2/figures/hinh_F7_heatmap_mau_theo_lop.pdf
Figure 9 values: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561/interpretation/report_heatmap_data.csv
Selected rows: T-shirt/top (4, 16, 18, 26) E=3.48; Trouser (2, 8, 9, 12) E=7.26; Pullover (2, 9, 20, 21) E=3.69; Dress (2, 4, 10, 28) E=3.17; Coat (2, 9, 11, 20, 27) E=3.12; Sandal (13, 19, 26) E=4.93; Shirt (9, 11, 16, 20) E=2.72; Sneaker (13, 21, 30) E=4.21; Bag (3, 31) E=4.88; Ankle boot (10, 12, 13) E=5.50


Process(`/usr/bin/python3 /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/src/applications/Fashion-Mnist/mining/scripts/report_figures.py --run-root /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/release/fashion_47265722452561 --report-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/reports/UD2`, ProcessExited(0))